<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 07 - Costes y Calidad de Datos

**Módulo:** Sistemas de Big Data (SBD - 5074)

**Contenido:** TCO, ROI, dimensiones de calidad de datos

---

## Objetivos
- Calcular el ROI de un proyecto de datos
- Identificar las 6 dimensiones de calidad de datos
- Implementar métricas de calidad automatizadas

## Las 6 Dimensiones de Calidad de Datos

| Dimensión | Definición | Métrica |
|-----------|------------|--------|
| **Completitud** | ¿Están todos los valores? | % no nulos |
| **Unicidad** | ¿Hay duplicados? | % registros únicos |
| **Exactitud** | ¿Son correctos los valores? | % valores válidos |
| **Consistencia** | ¿Son coherentes entre sí? | % sin contradicciones |
| **Validez** | ¿Cumplen el formato esperado? | % formato correcto |
| **Oportunidad** | ¿Están actualizados? | Antigüedad de datos |

## 1. Calculadora de calidad de datos

In [ ]:
import pandas as pd
import numpy as np

def calcular_calidad(df):
    """Calcula métricas de calidad para un DataFrame"""
    
    calidad = {}
    
    # Completitud: % de valores no nulos
    calidad['completitud'] = (df.notna().sum() / len(df) * 100).mean()
    
    # Unicidad: % sin duplicados
    duplicados = df.duplicated().sum()
    calidad['unicidad'] = (len(df) - duplicados) / len(df) * 100
    
    # Validez: depende de reglas específicas
    # Ejemplo para emails
    if 'email' in df.columns:
        patron = r'^[\w\.-]+@[\w\.-]+\.\w+$'
        validos = df['email'].str.match(patron, na=False).sum()
        calidad['validez_email'] = validos / len(df) * 100
    
    return calidad

# Ejemplo de uso
# Crear DataFrame de ejemplo con algunos problemas de calidad
np.random.seed(42)
df_ejemplo = pd.DataFrame({
    'id': [1, 2, 3, 4, 5, 5],  # Duplicado
    'nombre': ['Ana', 'Luis', None, 'María', 'Pedro', 'Pedro'],  # Nulo
    'email': ['ana@email.com', 'invalido', 'maria@test.es', None, 'pedro@mail.com', 'pedro@mail.com']
})

print("DataFrame de ejemplo:")
print(df_ejemplo)

# Calcular métricas de calidad
metricas = calcular_calidad(df_ejemplo)
print("\nMétricas de calidad:")
for metrica, valor in metricas.items():
    print(f"  {metrica}: {valor:.2f}%")

## 2. Análisis completo de calidad

In [ ]:
import random

import plotly.express as px
import plotly.graph_objects as go

class DataQualityAnalyzer:
    """Analizador completo de calidad de datos"""
    
    def __init__(self, df):
        self.df = df
        self.report = {}
    
    def analyze(self):
        """Ejecuta análisis completo"""
        self.report['completitud'] = self._completitud()
        self.report['unicidad'] = self._unicidad()
        self.report['tipos'] = self._tipos()
        return self.report
    
    def _completitud(self):
        """Analiza completitud por columna"""
        return {
            col: {
                'total': len(self.df),
                'no_nulos': self.df[col].notna().sum(),
                'nulos': self.df[col].isna().sum(),
                'porcentaje': self.df[col].notna().sum() / len(self.df) * 100
            }
            for col in self.df.columns
        }
    
    def _unicidad(self):
        """Analiza duplicados"""
        return {
            'total_filas': len(self.df),
            'filas_unicas': len(self.df.drop_duplicates()),
            'duplicados': self.df.duplicated().sum(),
            'porcentaje_unicos': (1 - self.df.duplicated().sum() / len(self.df)) * 100
        }
    
    def _tipos(self):
        """Analiza tipos de datos"""
        return {col: str(dtype) for col, dtype in self.df.dtypes.items()}
    
    def visualize(self):
        """Genera visualización de calidad"""
        if not self.report:
            self.analyze()
        
        # Gráfico de completitud
        completitud_df = pd.DataFrame([
            {'columna': col, 'completitud': data['porcentaje']}
            for col, data in self.report['completitud'].items()
        ])
        
        fig = px.bar(completitud_df, x='columna', y='completitud',
                     title='Completitud por Columna (%)',
                     color='completitud',
                     color_continuous_scale=['red', 'yellow', 'green'],
                     range_color=[0, 100])
        fig.add_hline(y=95, line_dash="dash", line_color="red",
                      annotation_text="Umbral 95%")
        return fig

# Crear dataset más grande con problemas de calidad
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'cliente_id': range(1, n+1),
    'nombre': [f'Cliente_{i}' if random.random() > 0.05 else None for i in range(n)],
    'email': [f'cliente{i}@mail.com' if random.random() > 0.1 else None for i in range(n)],
    'telefono': [f'6{random.randint(10000000, 99999999)}' if random.random() > 0.15 else None for i in range(n)],
    'edad': [random.randint(18, 80) if random.random() > 0.08 else None for i in range(n)],
    'ciudad': [random.choice(['Madrid', 'Barcelona', 'Sevilla', 'Valencia']) if random.random() > 0.03 else None for i in range(n)]
})

# Añadir algunos duplicados
duplicados = df.sample(50)
df = pd.concat([df, duplicados], ignore_index=True)

# Analizar
analyzer = DataQualityAnalyzer(df)
report = analyzer.analyze()

print("=== INFORME DE CALIDAD ===")
print(f"\nUnicidad: {report['unicidad']['porcentaje_unicos']:.2f}%")
print(f"Duplicados encontrados: {report['unicidad']['duplicados']}")

print("\nCompletitud por columna:")
for col, data in report['completitud'].items():
    estado = "✅" if data['porcentaje'] >= 95 else "⚠️" if data['porcentaje'] >= 80 else "❌"
    print(f"  {estado} {col}: {data['porcentaje']:.1f}%")

# Visualizar
fig = analyzer.visualize()
fig.show()

## 3. Calculadora de ROI

In [ ]:
def calcular_roi_proyecto_datos(costes, beneficios, años=3):
    """
    Calcula el ROI de un proyecto de datos
    
    Args:
        costes: dict con 'inicial' y 'anual'
        beneficios: dict con tipos de beneficio y valor anual
        años: horizonte temporal
    """
    
    coste_total = costes['inicial'] + (costes['anual'] * años)
    beneficio_total = sum(beneficios.values()) * años
    beneficio_neto = beneficio_total - coste_total
    roi = (beneficio_neto / coste_total) * 100
    
    return {
        'coste_total': coste_total,
        'beneficio_total': beneficio_total,
        'beneficio_neto': beneficio_neto,
        'roi': roi,
        'payback_meses': (costes['inicial'] / (sum(beneficios.values()) / 12)) if sum(beneficios.values()) > 0 else float('inf')
    }

# Ejemplo: Proyecto de Data Lake
costes = {
    'inicial': 150000,  # Implementación
    'anual': 50000      # Mantenimiento y cloud
}

beneficios = {
    'reduccion_costes_operativos': 80000,
    'aumento_ingresos_personalizacion': 45000,
    'reduccion_fraude': 30000,
    'mejora_eficiencia': 25000
}

resultado = calcular_roi_proyecto_datos(costes, beneficios, años=3)

print("=== ANÁLISIS ROI: Proyecto Data Lake ===")
print(f"\nCostes (3 años):")
print(f"  Inversión inicial: €{costes['inicial']:,.0f}")
print(f"  Coste anual: €{costes['anual']:,.0f}")
print(f"  Total: €{resultado['coste_total']:,.0f}")

print(f"\nBeneficios anuales:")
for beneficio, valor in beneficios.items():
    print(f"  {beneficio}: €{valor:,.0f}")
print(f"  Total anual: €{sum(beneficios.values()):,.0f}")

print(f"\n=== RESULTADOS ===")
print(f"  Beneficio total (3 años): €{resultado['beneficio_total']:,.0f}")
print(f"  Beneficio neto: €{resultado['beneficio_neto']:,.0f}")
print(f"  ROI: {resultado['roi']:.1f}%")
print(f"  Payback: {resultado['payback_meses']:.1f} meses")

---

## Recursos adicionales

- [Data Quality Dimensions](https://www.talend.com/resources/what-is-data-quality/)
- [Calculating ROI for Big Data](https://www.dataversity.net/calculating-roi-big-data/)